In [ ]:
import pandas as pd
import numpy as np

from snp_analysis_tools_sherlock import *
from coalescence_analysis_tools import *
import iqplot
import bokeh.plotting
import bokeh.io
import holoviews as hv
from holoviews import dim, opts
import bokeh.models
from bokeh.layouts import gridplot

hv.extension('bokeh')

In [ ]:
full_dfp7 = pd.read_csv('selection_coefficients_fitting2.csv').set_index('species-mesocosm')
full_dfp7 = pd.read_csv('selection_coefficients_fitting2_v2.csv').set_index('species-mesocosm')

full_dfp7 = full_dfp7.loc[~full_dfp7['parent_subjects'].isin(['AA-AC/PP',
                                                                    'AC/PP-AE','AC/PP-AF']),:]

e003_metadata = pd.read_csv('e003_metadata_cultures_round2.csv').drop(columns='Unnamed: 0')
full_dfp7['type_meso']

In [ ]:
full_dfp7['counts']=1
full_dfp7gr = full_dfp7.groupby(['species_id',
                  ]).sum(numeric_only=True
                        ).sort_values(by='counts',
                                      ascending=False)


cmap = bokeh.palettes.Light[9]
good_sp = full_dfp7gr.index.values[:len(cmap)]
cmap_sp = {}
for i, color in enumerate(cmap[:-1]):
    real_sp_name = full_dfp7.loc[full_dfp7['species_id'] == good_sp[i],'species'].unique()[0]
    print(real_sp_name)
    cmap_sp[real_sp_name] = cmap[i]
cmap_sp['other'] = cmap[-1]
cmap_sp

cmap_sp = {'s__Flavonifractor plautii': '#77AADD',
 's__Parabacteroides distasonis': '#EE8866',
 's__Bacteroides uniformis': '#FFAABB',
 's__Escherichia coli_D': '#EEDD88', 
 's__Bacteroides thetaiotaomicron': '#99DDFF',
 's__Dorea formicigenerans': '#44BB99',
 's__Parasutterella excrementihominis': '#BBCC33',
 's__Bacteroides_B dorei': '#AAAA00',
 'other': '#DDDDDD'}

full_dfp7['sp_plot'] = full_dfp7['species'].astype(str)
full_dfp7.loc[~full_dfp7['sp_plot'].isin(cmap_sp.keys()),'sp_plot'] = 'other'

In [ ]:
ep=1e-3
full_dfp7_gr_pol3 = full_dfp7.copy()
full_dfp7_gr_pol3.loc[full_dfp7_gr_pol3['p7_pred_frp3']<ep,'p7_pred_frp3'] = ep
full_dfp7_gr_pol3.loc[full_dfp7_gr_pol3['p7_pred_frp3']>1-ep,'p7_pred_frp3'] = 1-ep

full_dfp7_gr_pol3.loc[full_dfp7_gr_pol3['p0_freq']>.5,'strain_freq'] = 1-full_dfp7_gr_pol3.loc[full_dfp7_gr_pol3['p0_freq']>.5,'strain_freq']
full_dfp7_gr_pol3.loc[full_dfp7_gr_pol3['p0_freq']>.5,'p3_s'] = -full_dfp7_gr_pol3.loc[full_dfp7_gr_pol3['p0_freq']>.5,'p3_s']
full_dfp7_gr_pol3.loc[full_dfp7_gr_pol3['p0_freq']>.5,'p7_pred_frp3'] = 1-full_dfp7_gr_pol3.loc[full_dfp7_gr_pol3['p0_freq']>.5,'p7_pred_frp3']

full_dfp7_gr_pol3['p3_freq_fixed'] = False
full_dfp7_gr_pol3.loc[full_dfp7_gr_pol3['p3_freq']>=1-ep,'p3_freq_fixed']=True
full_dfp7_gr_pol3.loc[full_dfp7_gr_pol3['p3_freq']<=ep,'p3_freq_fixed']=True
#full_dfp7_nonzero_V2 = full_dfp7_nonzero.loc[full_dfp7_nonzero['parent_subjects'].isin(['AC/PP-AE','AA-AE','AA-AF']),:]
full_dfp7_gr_pol3['p3_s_gen'] = full_dfp7_gr_pol3['p3_s']/np.log2(200)
full_dfp7_gr_pol3['p7_s_fr5_gen'] = full_dfp7_gr_pol3['p7_s_fr5']/np.log2(200)
full_dfp7_gr_pol3['env'] = full_dfp7_gr_pol3['parent_media'] + '-' +  full_dfp7_gr_pol3['media']


In [ ]:
meds = full_dfp7.groupby(['type_meso','species', 'species_id','sp_plot']).median(numeric_only=True)
maxs = full_dfp7.groupby(['type_meso','species', 'species_id','sp_plot']).max(numeric_only=True)
mins =full_dfp7.groupby(['type_meso','species', 'species_id','sp_plot']).min(numeric_only=True)
meds['max_p7_pred_frp3'] = maxs['p7_pred_frp3']
meds['min_p7_pred_frp3'] = mins['p7_pred_frp3']
meds['max_strain_freq'] = maxs['strain_freq']
meds['min_strain_freq'] = mins['strain_freq']
meds = meds.reset_index()
#meds = meds.loc[meds['species_id'] == 101346,:]
scatter = hv.Scatter(meds.sort_values(by='sp_plot'), #.groupby(['type_meso','species', 'species_id','sp_plot']), #/.median(numeric_only=True).reset_index(),# .loc[full_dfp7_gr_pol3['species-type_meso']==type_meso_good,:],
#.loc[full_dfp7_gr_pol3['when_fixed']!=3,:], 
                     kdims =  'p7_pred_frp3', vdims=['strain_freq',hv.Dimension('sp_plot', )]).opts(#size=4, 
                                                                                    alpha = 1.0, color = 'sp_plot',#height=700,
                                                                                       width = 400,line_color='black',#height=900,
                                                                               
                                                                                                    
                                                                                                 cmap=cmap_sp,
                                                                                      # height = 350,
                                                                                    #  cmap = bokeh.palettes.Set3[11][::-1],
                                                                                                        colorbar=True, 
                                                                                             #  logx=True, logy=True,
    
    
    
                                                                                                legend_position='right',show_legend=False,
    
                                                                                                    size=10,
                                                                                         title = 'p7_sfr5 vs p5_s colored by p7_freq',
                                                                                                    xlim=(9e-4, 1-1e-4), ylim=(9e-4, 1-1e-4))
# (x0, y0, x1, y1)
seg = hv.Segments(meds, ['p7_pred_frp3', 
                         'min_strain_freq', 
                         'p7_pred_frp3', 'max_strain_freq', ]).opts(color='black',alpha=.2)

seg2 = hv.Segments(meds, ['min_p7_pred_frp3', 
                         'strain_freq', 
                         'max_p7_pred_frp3', 'strain_freq', ]).opts(color='black',alpha=.2)

scatter = hv.render(scatter*seg*seg2)
scatter.line(np.arange(-2000,2000)/1000, np.arange(-2000,2000)/1000,color = 'black', )
scatter.legend.visible = True
scatter.xaxis.axis_label = 'Predicted final frequency'
scatter.yaxis.axis_label = 'Actual final frequency'
scatter.title.text = 'Predicted final frequency based on first three passages'
bokeh.io.show(scatter)

In [ ]:
p3_s_gen#'] = full_dfp7_gr_pol3['p3_s']/np.log2(200)
full_dfp7_gr_pol3 #['p7_s_fr5_gen

In [ ]:
full_dfp7_gr = full_dfp7.groupby(['type_meso','species', 'species_id']).median(numeric_only=True)
full_dfp7_gr 

In [ ]:
full_dfp7_gr['pred_fix']= full_dfp7_gr['p7_pred_frp3']>=.95
full_dfp7_pred_fix = full_dfp7_gr.loc[full_dfp7_gr['pred_fix'],:] # we got 13
print(len(full_dfp7_pred_fix))
full_dfp7_pred_fix.loc[full_dfp7_pred_fix['strain_freq'] <.9,['strain_freq']] # we got 7 out of 21 

In [ ]:
full_dfp7_gr['pred_fix']= full_dfp7_gr['p7_pred_frp3']<=.05
full_dfp7_pred_fix = full_dfp7_gr.loc[full_dfp7_gr['pred_fix'],:] # we got 43 
print(len(full_dfp7_pred_fix))
full_dfp7_pred_fix.loc[full_dfp7_pred_fix['strain_freq'] >.1,['strain_freq']] # we got 7 out of 32

In [ ]:
full_dfp7['pred_fix']= full_dfp7['p7_pred_frp3']>=.95
full_dfp7_pred_fix = full_dfp7.loc[full_dfp7['pred_fix'],:] # we got 13
print(len(full_dfp7_pred_fix))
full_dfp7_pred_fix.loc[full_dfp7_pred_fix['strain_freq'] <.9,['strain_freq']] # we got 7 out of 21 
print(len(full_dfp7_pred_fix.loc[full_dfp7_pred_fix['strain_freq'] <.8,['strain_freq']]))

In [ ]:
full_dfp7['pred_fix']= full_dfp7['p7_pred_frp3']<=.05
full_dfp7_pred_fix = full_dfp7.loc[full_dfp7['pred_fix'],:] # we got 43 
print(len(full_dfp7_pred_fix))
full_dfp7_pred_fix.loc[full_dfp7_pred_fix['strain_freq'] >.2,['strain_freq']] # we got 7 out of 32
print(len(full_dfp7_pred_fix.loc[full_dfp7_pred_fix['strain_freq'] >.2,['strain_freq']]))

In [ ]:
(13+10)/(98+40)

In [ ]:
16+17

In [ ]:
39/138